# Pandas ETL Pipeline — Walley Risk Platform

**Purpose:** This notebook replicates a subset of the production PostgreSQL data-cleaning and feature-engineering pipeline (`sql/02_feature_engineering.sql`, `sql/03_data_cleaning.sql`) using **Python and Pandas**, to demonstrate an alternate ETL layer alongside the SQL implementation.

**Why both SQL and Pandas exist in this project:**
- **SQL** (`sql/` folder) is the production path — it runs the full cleaning and rule-flagging pipeline directly inside PostgreSQL at scale, close to the data, using window functions for velocity/mule detection that would be inefficient to replicate row-by-row in Python.
- **Pandas** (this notebook) demonstrates the same core transformation logic in a portable, dependency-light script — useful for ad-hoc analysis, local prototyping before promoting logic to SQL, or environments without direct database access.

**Scope:** This notebook works on a self-contained synthetic sample (generated inline below) so it can run standalone without a live PostgreSQL connection. The transformation *logic* mirrors the SQL scripts exactly; only the execution engine differs.

---
### Contents
1. Generate a synthetic sample (mirrors `generate_data.py`, smaller scale)
2. Data Cleaning — mirrors `03_data_cleaning.sql` (DAMA framework)
3. Feature Engineering — mirrors `02_feature_engineering.sql`
4. RegTech Rule Flagging — mirrors `04_regtech_rules.sql` (single-rule demo)
5. Data Quality Verification Report
6. Export cleaned dataset


In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

pd.set_option('display.max_columns', None)
np.random.seed(42)
random.seed(42)

print(f"pandas version: {pd.__version__}")
print(f"numpy version:  {np.__version__}")

pandas version: 3.0.2
numpy version:  2.4.4


## 1. Generate a Synthetic Sample

A smaller, self-contained sample (1,000 rows) is generated here so this notebook can run independently of the PostgreSQL database. Messy data is intentionally injected — negative amounts, malformed phone prefixes, nulls, and duplicate rows — to mirror the same data quality issues the SQL pipeline is built to catch.

In [2]:
N = 1000

# Hanoi-area coordinates as the "home" reference point, with jitter for realism
HOME_LAT, HOME_LON = 21.0278, 105.8342

def random_phone():
    prefix = random.choice(["+84", "84", "0"])
    number = "".join(str(random.randint(0, 9)) for _ in range(9))
    return f"{prefix}{number}"

def random_timestamp():
    start = datetime(2026, 6, 1)
    end = datetime(2026, 7, 22)
    delta = end - start
    random_seconds = random.randint(0, int(delta.total_seconds()))
    return start + timedelta(seconds=random_seconds)

data = {
    "transaction_id": range(1, N + 1),
    "user_id": np.random.randint(1000, 1200, size=N),
    "beneficiary_id": np.random.randint(5000, 5300, size=N),
    "amount": np.random.choice(
        [np.random.uniform(50_000, 8_000_000), np.random.uniform(9_000_000, 9_999_999),
         np.random.uniform(-500_000, -1)],  # negative values injected intentionally
        size=N, p=None
    ) if False else np.random.uniform(50_000, 15_000_000, size=N),
    "timestamp": [random_timestamp() for _ in range(N)],
    "phone_number": [random_phone() for _ in range(N)],
    "ip_latitude": HOME_LAT + np.random.uniform(-2, 2, size=N),
    "ip_longitude": HOME_LON + np.random.uniform(-2, 2, size=N),
    "home_latitude": HOME_LAT,
    "home_longitude": HOME_LON,
    "is_beneficiary_new": np.random.choice([True, False, None], size=N, p=[0.3, 0.5, 0.2]),
}

df = pd.DataFrame(data)

# Inject negative amounts (data quality issue #1) into ~3% of rows
neg_idx = df.sample(frac=0.03, random_state=1).index
df.loc[neg_idx, "amount"] = -df.loc[neg_idx, "amount"]

# Inject duplicate rows (data quality issue #2) — ~2% network-retry style duplicates
dupes = df.sample(frac=0.02, random_state=2)
df = pd.concat([df, dupes], ignore_index=True)

# Inject a few orphan rows with missing beneficiary_id (data quality issue #3)
orphan_idx = df.sample(frac=0.01, random_state=3).index
df.loc[orphan_idx, "beneficiary_id"] = np.nan

print(f"Raw sample shape: {df.shape}")
df.head()

Raw sample shape: (1020, 11)


,transaction_id,user_id,beneficiary_id,amount,timestamp,phone_number,ip_latitude,ip_longitude,home_latitude,home_longitude,is_beneficiary_new
0,1,1102,5235.0,2.762503e+06,2026-06-11 19:25:12,+84004711382,22.041512,106.530177,21.0278,105.8342,True
1,2,1179,5082.0,7.901193e+06,2026-06-03 10:16:45,84758692617,22.103491,104.084459,21.0278,105.8342,True
2,3,1092,5041.0,1.065024e+07,2026-06-27 16:51:53,0640537735,21.661962,104.676228,21.0278,105.8342,True
3,4,1014,5100.0,1.647810e+06,2026-06-24 18:38:21,+84585064317,22.092264,107.070295,21.0278,105.8342,False
4,5,1106,5005.0,8.531318e+06,2026-06-22 16:07:07,+84390053293,22.411486,104.421888,21.0278,105.8342,True


## 2. Data Cleaning — DAMA Framework (mirrors `03_data_cleaning.sql`)

Same five DAMA dimensions applied in the SQL pipeline, implemented here in Pandas.

### 2.1 Accuracy & Conformity — phone number standardization

SQL equivalent:
```sql
-- Standardizing +84 / 84 prefixes to 0
UPDATE transactions
SET phone_number = REGEXP_REPLACE(phone_number, '^(\+84|84)', '0');
```

In [3]:
df["phone_number"] = df["phone_number"].str.replace(r'^(\+84|84)', '0', regex=True)
df["phone_number"].sample(10, random_state=1)

267    0899217223
142    0772522872
839    0078263337
175    0571775140
481    0205554384
320    0775170166
578    0063461571
94     0952694958
104    0157473384
49     0412478261
Name: phone_number, dtype: str

### 2.2 Validity — correcting negative amounts

SQL equivalent:
```sql
UPDATE transactions SET amount = ABS(amount) WHERE amount < 0;
```

In [4]:
negative_count = (df["amount"] < 0).sum()
df["amount"] = df["amount"].abs()
print(f"Corrected {negative_count} negative amount(s) via abs().")

Corrected 30 negative amount(s) via abs().


### 2.3 Completeness — NULL imputation

SQL equivalent:
```sql
-- Conservative default: treat missing beneficiary status as "new" (higher risk stance)
UPDATE transactions
SET is_beneficiary_new = COALESCE(is_beneficiary_new, TRUE);
```

The same conservative-default principle applies here: an unknown beneficiary status is treated as "new" rather than silently dropped, since under-flagging a potentially risky transaction is worse than a false positive in a fraud-detection context.

In [5]:
null_count = df["is_beneficiary_new"].isna().sum()
df["is_beneficiary_new"] = df["is_beneficiary_new"].fillna(True)
print(f"Imputed {null_count} null is_beneficiary_new value(s) with conservative default (True).")

Imputed 204 null is_beneficiary_new value(s) with conservative default (True).


### 2.4 Uniqueness — removing duplicate transactions

SQL equivalent:
```sql
DELETE FROM transactions
WHERE transaction_id NOT IN (
    SELECT transaction_id FROM (
        SELECT transaction_id,
               ROW_NUMBER() OVER (PARTITION BY user_id, beneficiary_id, amount, timestamp ORDER BY transaction_id) AS rn
        FROM transactions
    ) t WHERE rn = 1
);
```

`ROW_NUMBER() OVER (PARTITION BY ...)` keeping only `rn = 1` is functionally identical to Pandas' `drop_duplicates(subset=..., keep='first')`.

In [6]:
before = len(df)
df = df.drop_duplicates(subset=["user_id", "beneficiary_id", "amount", "timestamp"], keep="first")
after = len(df)
print(f"Removed {before - after} duplicate transaction(s). Rows remaining: {after}")

Removed 20 duplicate transaction(s). Rows remaining: 1000


### 2.5 Referential Integrity — purging orphan transactions

SQL equivalent:
```sql
DELETE FROM transactions t
WHERE NOT EXISTS (SELECT 1 FROM beneficiaries b WHERE b.beneficiary_id = t.beneficiary_id);
```

In [7]:
before = len(df)
df = df.dropna(subset=["beneficiary_id", "user_id"])
after = len(df)
print(f"Purged {before - after} orphan transaction(s) with missing user_id/beneficiary_id.")
df["beneficiary_id"] = df["beneficiary_id"].astype(int)

Purged 10 orphan transaction(s) with missing user_id/beneficiary_id.


## 3. Feature Engineering (mirrors `02_feature_engineering.sql`)

### 3.1 Geolocation distance — Haversine formula

SQL equivalent:
```sql
UPDATE transactions t
SET distance_from_home_km = (
    6371 * acos(
        least(1.0, greatest(-1.0,
            cos(radians(u.home_latitude)) * cos(radians(t.ip_latitude)) *
            cos(radians(t.ip_longitude) - radians(u.home_longitude)) +
            sin(radians(u.home_latitude)) * sin(radians(t.ip_latitude))
        ))
    )
)
FROM users u WHERE t.user_id = u.user_id;
```

The `np.clip(..., -1.0, 1.0)` call below is the direct Pandas/Numpy equivalent of the SQL `least(1.0, greatest(-1.0, ...))` guard — both exist to prevent a domain error in `acos()` when floating-point rounding pushes the input slightly outside its valid [-1, 1] range.

In [8]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    cos_val = (
        np.cos(lat1) * np.cos(lat2) * np.cos(lon2 - lon1)
        + np.sin(lat1) * np.sin(lat2)
    )
    cos_val = np.clip(cos_val, -1.0, 1.0)  # guards against acos() domain errors, same as SQL's least/greatest clamp
    return 6371 * np.arccos(cos_val)

df["distance_from_home_km"] = haversine_km(
    df["home_latitude"], df["home_longitude"],
    df["ip_latitude"], df["ip_longitude"]
)

df["distance_from_home_km"].describe()

count    990.000000
mean     163.418115
std       60.856393
min       12.990009
25%      122.077173
50%      166.018756
75%      208.737958
max      301.874513
Name: distance_from_home_km, dtype: float64

### 3.2 Temporal context — off-hours and weekend flags

SQL equivalent:
```sql
UPDATE transactions
SET is_off_hours = EXTRACT(HOUR FROM timestamp) >= 23 OR EXTRACT(HOUR FROM timestamp) < 5,
    is_weekend   = EXTRACT(DOW FROM timestamp) IN (0, 6);
```

In [9]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["is_off_hours"] = df["timestamp"].dt.hour.apply(lambda h: h >= 23 or h < 5)
df["is_weekend"] = df["timestamp"].dt.dayofweek.isin([5, 6])  # Saturday=5, Sunday=6

df[["timestamp", "is_off_hours", "is_weekend"]].sample(10, random_state=1)

,timestamp,is_off_hours,is_weekend
323,2026-07-05 17:45:21,False,True
844,2026-07-15 05:01:47,False,False
956,2026-06-13 18:12:42,False,True
711,2026-07-10 15:54:53,False,False
951,2026-06-04 19:51:40,False,False
990,2026-06-23 16:50:40,False,False
489,2026-06-21 01:57:32,True,True
987,2026-07-07 23:51:13,True,False
910,2026-06-14 21:29:18,False,True
680,2026-07-09 21:42:57,False,False


## 4. RegTech Rule Flagging — Biometric Evasion Demo (mirrors `04_regtech_rules.sql`)

Single-rule demonstration in Pandas. The full multi-rule engine (velocity bursts, mule networks) remains in SQL, where window functions and self-joins handle time-series and graph-style logic more efficiently than an equivalent Pandas rolling/merge implementation would at scale.

SQL equivalent:
```sql
UPDATE transactions t
SET rule_flags = array_append(rule_flags, 'biometric_evasion_rule')
FROM beneficiaries b
WHERE t.beneficiary_id = b.beneficiary_id
  AND t.amount BETWEEN 9000000 AND 9999999
  AND COALESCE(t.is_beneficiary_new, TRUE) = TRUE;
```

In [10]:
df["biometric_evasion_flag"] = (
    df["amount"].between(9_000_000, 9_999_999)
    & (df["is_beneficiary_new"] == True)
)

flagged_count = df["biometric_evasion_flag"].sum()
flag_rate = flagged_count / len(df) * 100
print(f"Biometric evasion pattern matched: {flagged_count} transactions ({flag_rate:.2f}% of sample)")

df[df["biometric_evasion_flag"]][["transaction_id", "amount", "is_beneficiary_new", "timestamp"]].head()

Biometric evasion pattern matched: 49 transactions (4.95% of sample)


,transaction_id,amount,is_beneficiary_new,timestamp
13,14,9.044097e+06,True,2026-06-23 14:08:21
30,31,9.376697e+06,True,2026-06-10 00:06:48
39,40,9.647474e+06,True,2026-07-07 18:02:56
40,41,9.126056e+06,True,2026-06-08 15:37:03
73,74,9.780617e+06,True,2026-07-09 22:46:54


## 5. Data Quality Verification Report

Mirrors the verification query at the end of `03_data_cleaning.sql`, confirming zero invalid values remain post-cleaning.

In [11]:
report = {
    "cleaned_transactions": len(df),
    "invalid_amounts_left": int((df["amount"] < 0).sum()),
    "null_distances_left": int(df["distance_from_home_km"].isna().sum()),
    "orphan_txns_left": int(df["beneficiary_id"].isna().sum() + df["user_id"].isna().sum()),
    "duplicate_txns_left": int(df.duplicated(subset=["user_id", "beneficiary_id", "amount", "timestamp"]).sum()),
}

print("=== DATA CLEANING REPORT (Pandas ETL) ===")
for k, v in report.items():
    print(f"{k:<25}: {v}")

=== DATA CLEANING REPORT (Pandas ETL) ===
cleaned_transactions     : 990
invalid_amounts_left     : 0
null_distances_left      : 0
orphan_txns_left         : 0
duplicate_txns_left      : 0


## 6. Export Cleaned Dataset

Exports the cleaned, feature-engineered sample to CSV — usable as a Power BI or Excel data source, or for further analysis.

In [12]:
output_path = "cleaned_transactions_pandas_etl.csv"
df.to_csv(output_path, index=False)
print(f"Exported {len(df)} cleaned rows to {output_path}")

Exported 990 cleaned rows to cleaned_transactions_pandas_etl.csv


---
## Summary

| Step | SQL Implementation | Pandas Equivalent |
|---|---|---|
| Phone standardization | `REGEXP_REPLACE` | `str.replace()` with regex |
| Negative amount correction | `ABS()` | `.abs()` |
| Null imputation | `COALESCE(x, TRUE)` | `.fillna(True)` |
| Deduplication | `ROW_NUMBER() OVER (PARTITION BY ...)` | `.drop_duplicates(subset=...)` |
| Referential integrity | `NOT EXISTS` subquery | `.dropna(subset=...)` |
| Haversine distance | `acos()`/`radians()` with `least/greatest` clamp | `np.arccos()`/`np.radians()` with `np.clip()` |
| Off-hours/weekend flags | `EXTRACT(HOUR/DOW FROM timestamp)` | `.dt.hour`, `.dt.dayofweek` |
| Rule flagging | `UPDATE ... WHERE ... BETWEEN` | Boolean mask with `.between()` |

**Takeaway:** the same data quality and feature engineering logic is portable across both engines. SQL remains the production choice for this project given the scale (18K+ rows) and the window-function-heavy velocity/mule detection rules; Pandas serves as a lightweight, dependency-minimal alternative for prototyping or environments without direct database access.
